In [1]:
import re, requests
from urllib.parse import unquote

QLEVER = "https://qlever.dev/api/wikidata"
QHEADERS = {"Accept": "text/tab-separated-values", "Content-type": "application/sparql-query"}

def qlever(query):
    """POST SPARQL -> list of dict rows. QLever returns TSV: IRIs as <...>, literals quoted."""
    r = requests.post(QLEVER, headers=QHEADERS, data=query.encode(), timeout=60)
    r.raise_for_status()
    rows = [ln for ln in r.text.split("\n") if ln.strip()]
    cols = [c.lstrip("?") for c in rows[0].split("\t")]
    def clean(c):
        c = c.strip()
        if c[:1] == "<" and c[-1:] == ">": return c[1:-1]
        if c[:1] == '"': return c[1:c.rfind('"')]
        return c or None
    return [dict(zip(cols, [clean(x) for x in ln.split("\t")])) for ln in rows[1:]]

qid = lambda u: u.rsplit("/", 1)[-1] if u else None
IMDB_ID = "tt0903747"  # Breaking Bad

In [2]:
# 1. Spine: single row. Look at the raw values first.
spine = qlever(f"""
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX schema: <http://schema.org/>
SELECT ?rtId ?enwiki WHERE {{
  ?item wdt:P345 "{IMDB_ID}" .
  OPTIONAL {{ ?item wdt:P1258 ?rtId . }}
  OPTIONAL {{ ?enwiki schema:about ?item ; schema:isPartOf <https://en.wikipedia.org/> . }}
}}""")
print("raw spine row:", spine[0])
rt_url = "https://www.rottentomatoes.com/" + spine[0]["rtId"] if spine[0].get("rtId") else None
print("RT       :", rt_url)
print("Wikipedia:", spine[0].get("enwiki"))

raw spine row: {'rtId': 'tv/breaking_bad', 'enwiki': 'https://en.wikipedia.org/wiki/Breaking_Bad'}
RT       : https://www.rottentomatoes.com/tv/breaking_bad
Wikipedia: https://en.wikipedia.org/wiki/Breaking_Bad


In [3]:
stmts = qlever(f"""
PREFIX wdt:<http://www.wikidata.org/prop/direct/>
PREFIX p:<http://www.wikidata.org/prop/>
PREFIX ps:<http://www.wikidata.org/prop/statement/>
PREFIX pq:<http://www.wikidata.org/prop/qualifier/>
SELECT ?result ?level ?award ?year WHERE {{
  ?film wdt:P345 "{IMDB_ID}" .
  {{ {{ ?film p:P166 ?st . ?st ps:P166 ?award . BIND("title" AS ?level) }}
     UNION {{ ?person p:P166 ?st . ?st ps:P166 ?award ; pq:P1686 ?film . BIND("person" AS ?level) }}
     BIND("won" AS ?result) }}
  UNION
  {{ {{ ?film p:P1411 ?st . ?st ps:P1411 ?award . BIND("title" AS ?level) MINUS {{ ?film wdt:P166 ?award . }} }}
     UNION {{ ?person p:P1411 ?st . ?st ps:P1411 ?award ; pq:P1686 ?film . BIND("person" AS ?level)
              MINUS {{ ?person p:P166 ?w . ?w ps:P166 ?award ; pq:P1686 ?film . }} }}
     BIND("nominated" AS ?result) }}
  OPTIONAL {{ ?st pq:P585 ?t . BIND(YEAR(?t) AS ?year) }}
}}""")
print("raw award rows (first 3):")
for s in stmts[:3]:
    print("  ", s)
won = sum(s["result"] == "won" for s in stmts)
nom = sum(s["result"] == "nominated" for s in stmts)
print(f"\nwon: {won} | nominated: {nom} | total: {won + nom}")

raw award rows (first 3):
   {'result': 'nominated', 'level': 'person', 'award': 'http://www.wikidata.org/entity/Q8038466', 'year': '2011'}
   {'result': 'nominated', 'level': 'person', 'award': 'http://www.wikidata.org/entity/Q8038466', 'year': '2010'}
   {'result': 'nominated', 'level': 'title', 'award': 'http://www.wikidata.org/entity/Q7243502', 'year': '2013'}

won: 60 | nominated: 36 | total: 96


In [4]:
# Award labels come back as bare QIDs above; resolve them in one small VALUES query.
award_qids = {qid(s["award"]) for s in stmts if (qid(s["award"]) or "").startswith("Q")}
vals = " ".join(f"wd:{q}" for q in award_qids)
labels = {qid(x["a"]): x["aLabel"] for x in qlever(
    "PREFIX wd:<http://www.wikidata.org/entity/>\n"
    "PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>\n"
    f"SELECT ?a ?aLabel WHERE {{ VALUES ?a {{ {vals} }} ?a rdfs:label ?aLabel . FILTER(LANG(?aLabel)='en') }}")}

from collections import Counter
fam = Counter(labels.get(qid(s["award"]), "?").split(" for ")[0].split(" Award")[0] for s in stmts)
print("top award families:", fam.most_common(6))

top award families: [('Primetime Emmy', 49), ('Screen Actors Guild', 9), ('Satellite', 9), ('Saturn', 8), ('Writers Guild of America', 6), ('TCA', 6)]


In [5]:
# ONE request: TextExtracts returns the WHOLE article as plain text.
# exsectionformat="wiki" makes headings come back as "== Plot ==", so we slice locally.
WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_HEADERS = {"User-Agent": "CineSyncDemo/0.1 (personal project; demo notebook)"}  # Wikipedia rejects UA-less requests
PLOT_HEADINGS = {"plot", "plot summary", "synopsis", "premise", "plot synopsis"}

page = unquote(spine[0]["enwiki"].rsplit("/", 1)[-1])
data = requests.get(WIKI_API, headers=WIKI_HEADERS, params={
    "action": "query", "prop": "extracts", "explaintext": "1",
    "exsectionformat": "wiki", "redirects": "1", "format": "json", "titles": page,
}, timeout=60).json()
extract = next(iter(data["query"]["pages"].values()))["extract"]
print("whole-article extract:", len(extract), "chars")
print("headings:", re.findall(r"(?m)^=+\s*(.+?)\s*=+$", extract)[:8], "...")


# Slice the Plot section: from its heading down to the next same-or-higher-level heading
# (so plot subsections are kept). This mirrors production's wikipedia.slice_plot.
def slice_plot(text):
    heads = list(re.finditer(r"(?m)^(={2,6})\s*(.+?)\s*\1\s*$", text))
    for i, h in enumerate(heads):
        if h.group(2).strip().lower() in PLOT_HEADINGS:
            level = len(h.group(1))
            end = next((n.start() for n in heads[i + 1:] if len(n.group(1)) <= level), len(text))
            return text[h.end():end].strip() or None
    return None


plot = slice_plot(extract)
print("\nplot length:", len(plot or ""), "chars\n")
print((plot or "")[:700], "...")

whole-article extract: 73408 chars
headings: ['Premise', 'Cast and characters', 'Main characters', 'Recurring characters', 'Production', 'Conception', 'Development history', 'Writing'] ...

plot length: 870 chars

Breaking Bad follows Walter White, a financially struggling high school chemistry teacher from Albuquerque, New Mexico, who enters the local methamphetamine trade after being diagnosed with stage-three lung cancer. Seeking to secure his family's financial future, Walter begins producing methamphetamine with his former student Jesse Pinkman in a rolling meth lab. Their operation gradually expands as they produce a highly pure form of blue methamphetamine that becomes widely sought after. Walter adopts the alias "Heisenberg" to conceal his identity. His involvement in the drug trade brings him into conflict with his family, the Drug Enforcement Administration (DEA) through his brother-in-law H ...
